# Mapping the Language Family Tree

How big are the world's language families, and how deep does German's lineage go?
This notebook explores Glottolog's phylogeny via `db.families` and walks the
ancestor chain for individual languages.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [1]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [2]:
db = low.LanguagesOfTheWorld()
print(db)
print(f"Family tree nodes: {len(db.families)}")

LanguagesOfTheWorld(languages=7929, countries=247, continents=5, scripts=106, speaker_counts=8969, language_names=167917)
Family tree nodes: 4853


## 2 · Top-level root families

`db.families.roots()` returns families with no parent — the trunks of the tree.

In [3]:
roots = db.families.roots()
print(f"Root families: {len(roots)}")
for fam in sorted(roots, key=lambda f: f.label)[:8]:
    print(f"  {fam.label} ({fam.glottocode}) — {len(fam.children)} direct subgroups")

Root families: 246
  Abkhaz-Adyge (abkh1242) — 2 direct subgroups
  Afro-Asiatic (afro1255) — 5 direct subgroups
  Ainu (ainu1252) — 1 direct subgroups
  Algic (algi1248) — 1 direct subgroups
  Amto-Musan (amto1249) — 0 direct subgroups
  Angan (anga1289) — 1 direct subgroups
  Anim (anim1240) — 3 direct subgroups
  Arafundi (araf1243) — 1 direct subgroups


## 3 · Count descendant languages per root family

In [4]:
def count_descendant_languages(family, seen=None):
    """Recursively count all languages under a family node."""
    seen = seen or set()
    if family.glottocode in seen:
        return 0
    seen.add(family.glottocode)
    total = len(family.languages)
    for child in family.children:
        total += count_descendant_languages(child, seen)
    return total

root_rows = [
    {
        "family": fam.label,
        "glottocode": fam.glottocode,
        "subgroups": len(fam.children),
        "languages": count_descendant_languages(fam),
    }
    for fam in roots
]

root_df = pd.DataFrame(root_rows).sort_values("languages", ascending=False)
root_df.head(15)

,family,glottocode,subgroups,languages
13,Atlantic-Congo,atla1278,5,1344
15,Austronesian,aust1307,6,1243
78,Indo-European,indo1319,3,548
189,Sino-Tibetan,sino1245,18,440
1,Afro-Asiatic,afro1255,5,370
161,Nuclear Trans New Guinea,nucl1709,10,311
169,Pama-Nyungan,pama1250,18,211
166,Otomanguean,otom1299,2,178
188,Sign Language,sign1238,3,156
14,Austroasiatic,aust1305,12,152


## 4 · Treemap of top families by language count

In [5]:
top_roots = root_df.head(20)

fig = px.treemap(
    top_roots,
    path=["family"],
    values="languages",
    title="Top 20 Root Language Families by Descendant Language Count",
    color="languages",
    color_continuous_scale="Blues",
)
fig.update_layout(margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 5 · Walk German's lineage

Starting from `Language.family`, follow `.parent` links up to the root.
`depth` and `ancestors` provide the same information as properties.

In [6]:
deu = db.languages.get("deu")
print(f"{deu.label} — glottocode: {deu.glottocode}")
print(f"Immediate family: {deu.family.label}")
print()
print("Lineage (leaf → root):")
node = deu.family
while node:
    indent = "  " * node.depth
    print(f"{indent}{node.label} (depth={node.depth})")
    node = node.parent

print()
print("Via .ancestors property:")
for anc in deu.family.ancestors:
    print(f"  {anc.label}")
print(f"Root family: {deu.family.root.label}")

German — glottocode: stan1295
Immediate family: Global German

Lineage (leaf → root):
                    Global German (depth=10)
                  Upper Franconian (depth=9)
                Modern High German (depth=8)
              Middle-Modern High German (depth=7)
            Upper German (depth=6)
          High German (depth=5)
        West Germanic (depth=4)
      Northwest Germanic (depth=3)
    Germanic (depth=2)
  Classical Indo-European (depth=1)
Indo-European (depth=0)

Via .ancestors property:
  Upper Franconian
  Modern High German
  Middle-Modern High German
  Upper German
  High German
  West Germanic
  Northwest Germanic
  Germanic
  Classical Indo-European
  Indo-European
Root family: Indo-European


## 6 · Lookup by glottocode or label

In [7]:
indo = db.families.get("indo1319")
print(f"get('indo1319') → {indo.label}")
print(f"Direct child languages: {len(indo.languages)}")
print(f"Direct subgroups: {len(indo.children)}")
for child in indo.children[:5]:
    print(f"  {child.label} ({len(child.languages)} langs, {len(child.children)} subgroups)")

get('indo1319') → Indo-European
Direct child languages: 0
Direct subgroups: 3
  Anatolian (1 langs, 1 subgroups)
  Classical Indo-European (0 langs, 9 subgroups)
  Unclassified Indo-European (3 langs, 0 subgroups)


## 7 · Families with the most immediate member languages

In [8]:
immediate = sorted(
    (f for f in db.families if f.languages),
    key=lambda f: len(f.languages),
    reverse=True,
)[:15]

imm_df = pd.DataFrame([
    {"family": f.label, "glottocode": f.glottocode, "immediate_languages": len(f.languages)}
    for f in immediate
])

fig2 = px.bar(
    imm_df,
    x="immediate_languages",
    y="family",
    orientation="h",
    title="Families with Most Direct Member Languages",
    labels={"immediate_languages": "Languages", "family": ""},
)
fig2.update_layout(yaxis={"categoryorder": "total ascending"}, height=500)
fig2.show()

## 8 · Summary

In [9]:
kin = db.languages.get("kin")
print(f"{kin.label}: family={kin.family.label if kin.family else '—'}, endangerment={kin.endangerment}")
print(f"Total languages with a family assignment: {sum(1 for l in db.languages if l.family):,}")

Kinyarwanda: family=West Highlands Kivu, endangerment=not_endangered
Total languages with a family assignment: 7,388
